# 02 · Limpieza de Datos

> **Objetivo:** convertir el dataset crudo en un conjunto consistente
> y confiable para el feature engineering.

## ¿Qué vamos a hacer?

1. Eliminar filas sin `CustomerID`.
2. Eliminar cancelaciones.
3. Filtrar `Quantity > 0` y `Price > 0`.
4. Eliminar duplicados exactos.
5. Calcular `LineTotal = Quantity * Price`.
6. Guardar el resultado en `data/interim/transacciones_limpias.parquet`.

## Concepto teórico: las tres opciones de limpieza

Frente a un dato problemático tienes **tres** opciones, y elegir bien
es lo que separa a un científico de datos junior de uno senior:

| Opción | Cuándo aplica | Riesgos |
|---|---|---|
| **Eliminar** | Cuando el dato está irrecuperablemente roto o no se puede atribuir. | Sesgo si los faltantes no son aleatorios. |
| **Imputar** | Cuando hay un valor "por defecto" razonable (mediana, moda). | Aplanar la varianza, inventar señal. |
| **Transformar** | Cuando el dato es real pero "extremo" (outliers, sesgo). | Distorsionar la interpretación. |

> **Regla de oro:** documenta CADA decisión. En entrevistas y auditorías
> te van a preguntar el *por qué*, no el *qué*.


In [ ]:
# Permite importar el paquete src/ desde el notebook
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import pandas as pd

from src.data.loader import load_raw_transactions
from src.features.rfm import clean_transactions
from src.config import CLEAN_DATA_FILE

pd.set_option("display.max_columns", 30)


## 1. Cargar el dataset crudo

In [ ]:
df = load_raw_transactions()
print(f"Filas iniciales: {len(df):,}")


## 2. Aplicar las reglas de limpieza

La función ``clean_transactions`` aplica las cinco reglas en orden y
devuelve el dataframe limpio. Vamos a verla en acción y luego desglosarla
manualmente para entender cada paso.


In [ ]:
df_clean = clean_transactions(df)
print(f"Filas después de limpieza: {len(df_clean):,}")
df_clean.head()


## 3. Desglose paso a paso (didáctico)

Para entender *qué* hace `clean_transactions`, repliquemos sus pasos.


In [ ]:
n0 = len(df)
print(f"0. Inicial: {n0:,}")

# Paso 1: eliminar sin CustomerID
df_step = df.dropna(subset=["CustomerID"]).copy()
print(f"1. Sin CustomerID NaN: {len(df_step):,} (-{n0 - len(df_step):,})")

# Paso 2: eliminar cancelaciones
df_step["Invoice"] = df_step["Invoice"].astype(str)
n_prev = len(df_step)
df_step = df_step[~df_step["Invoice"].str.startswith("C")]
print(f"2. Sin cancelaciones: {len(df_step):,} (-{n_prev - len(df_step):,})")

# Paso 3: cantidades y precios positivos
n_prev = len(df_step)
df_step = df_step[(df_step["Quantity"] > 0) & (df_step["Price"] > 0)]
print(f"3. Quantity y Price > 0: {len(df_step):,} (-{n_prev - len(df_step):,})")

# Paso 4: duplicados
n_prev = len(df_step)
df_step = df_step.drop_duplicates()
print(f"4. Sin duplicados exactos: {len(df_step):,} (-{n_prev - len(df_step):,})")


> **Tip:** mantener un log con el conteo en cada paso es invaluable.
> Si mañana cambia el dataset y los números bajan demasiado, sabes
> exactamente en qué paso revisar.

## 4. Validación post-limpieza


In [ ]:
checks = {
    "CustomerID nulos": df_clean["CustomerID"].isna().sum(),
    "Cancelaciones (C*)": df_clean["Invoice"].astype(str).str.startswith("C").sum(),
    "Quantity <= 0": (df_clean["Quantity"] <= 0).sum(),
    "Price <= 0": (df_clean["Price"] <= 0).sum(),
    "Duplicados": df_clean.duplicated().sum(),
}
for k, v in checks.items():
    status = "OK" if v == 0 else "FALLA"
    print(f"  [{status}] {k}: {v}")


## 5. Detección de outliers (sin eliminarlos)

A diferencia de las cuatro reglas anteriores, los outliers en `Quantity`
y `Monetary` **no los eliminamos**. Razones:

1. Pueden ser **clientes mayoristas legítimos**, que son un segmento de negocio importante.
2. La transformación logarítmica que aplicaremos en los pipelines reduce su influencia.
3. Eliminar outliers a ciegas puede esconder el segmento más rentable.

Veamos su magnitud:


In [ ]:
df_clean["LineTotal"] = df_clean["Quantity"] * df_clean["Price"]

p99 = df_clean[["Quantity", "Price", "LineTotal"]].quantile([0.5, 0.95, 0.99, 1.0])
print("Percentiles de variables continuas:")
print(p99)


> **Observa** la diferencia entre la mediana y el máximo. Eso es un
> long-tail clásico.

## 6. Guardar el resultado

Lo guardamos en `parquet` (más compacto y rápido que CSV).


In [ ]:
df_clean.to_parquet(CLEAN_DATA_FILE, index=False)
print(f"Guardado en: {CLEAN_DATA_FILE}")
print(f"Tamaño: {CLEAN_DATA_FILE.stat().st_size / 1e6:.2f} MB")


## Resumen

| Decisión | Razón |
|---|---|
| Eliminar `CustomerID` NaN | No se pueden atribuir; imputar inventaría. |
| Eliminar cancelaciones | No son compras netas. |
| Filtrar `Quantity` y `Price` > 0 | Errores de captura. |
| Eliminar duplicados exactos | Errores de carga. |
| **Conservar outliers** | Pueden ser mayoristas → segmento valioso. |

---

## Preguntas de Reflexión

1. Si hicieras el RFM **restando** las cancelaciones (en lugar de
   eliminarlas), ¿qué cambios verías en el segmento "Champions"?
2. ¿Por qué `parquet` es preferible a `csv` para datos numéricos grandes?
3. ¿Qué pasaría si un cliente mayorista hace una sola gran compra?
   ¿En qué cluster terminaría con K-Means? ¿Y con DBSCAN?

> **Próximo paso:** ``03_feature_engineering.ipynb`` — convertir las
> transacciones en perfiles RFM por cliente.
